In [18]:
# Import Libraries & Load Data

# Import core libraries for data manipulation and analysis
import pandas as pd
import numpy as np

# Load the training dataset
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

In [19]:
# Feature Engineering & Data Cleaning

# Define a helper function to categorize family size into logical groups
def group_family(size):
    if size == 1:
        return 'Alone'
    elif size <= 4:
        return 'Small'
    else:
        return 'Large'

# --- 1. Title Extraction ---
# Extract titles from raw name string
train_df['Title'] = train_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Group rare titles into a single category
train_df['Title'] = train_df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
train_df['Title'] = train_df['Title'].replace('Mlle', 'Miss')
train_df['Title'] = train_df['Title'].replace('Ms', 'Miss')
train_df['Title'] = train_df['Title'].replace('Mme', 'Mrs')

# Map titles to distinct numerical values
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
train_df['Title'] = train_df['Title'].map(title_mapping)
train_df['Title'] = train_df['Title'].fillna(0)

# --- 2. Family Grouping ---
# Calculate family size and map into Alone, Small, or Large groups
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
train_df['FamilyGroup'] = train_df['FamilySize'].apply(group_family)

# Encode family groups numerically based on non-linear survival logic
family_mapping = {'Small': 1, 'Alone': 2, 'Large': 3}
train_df['FamilyGroup'] = train_df['FamilyGroup'].map(family_mapping)

# --- 3. Handling Missing Values & Encoding ---
# Fill missing age values with median
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())

# Fill missing embarkation values with mode
train_df['Embarked'] = train_df['Embarked'].fillna(train_df['Embarked'].mode()[0])

# Map categorical text features to numerical labels
train_df['Sex'] = train_df['Sex'].map({'male': 0, 'female': 1})
train_df['Embarked'] = train_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Feature Enrichment from String Data
for df in [train_df, test_df]:
    # 1. Deck Extraction
    # Fill missing values with 'U' (Unknown) and extract the first letter to determine the deck level
    df['Deck'] = df['Cabin'].fillna('U').apply(lambda x: str(x)[0])
    
    # 2. Ticket Frequency (Group Size)
    # Count the number of passengers sharing the exact same ticket number to identify group size
    df['Ticket_Freq'] = df.groupby('Ticket')['Ticket'].transform('count')

# Drop the original 'Cabin' and 'Ticket' columns since their key information has already been extracted
train_df = train_df.drop(['Cabin', 'Ticket'], axis=1)
test_df = test_df.drop(['Cabin', 'Ticket'], axis=1)

In [20]:
# Categorical Encoding & Data Alignment

# 1. Apply One-Hot Encoding to the new 'Deck' column
train_df = pd.get_dummies(train_df, columns=['Deck'])
test_df = pd.get_dummies(test_df, columns=['Deck'])

# 2. Align the test dataset to match the training dataset columns
# This step ensures that if a deck (e.g., Deck_T) exists in train but not in test,
# the column will be created in the test set and filled with 0s.
missing_cols = set(train_df.columns) - set(test_df.columns)
for col in missing_cols:
    if col != 'Survived':  # Exclude the target variable
        test_df[col] = 0

# 3. Ensure the column order in test_df strictly matches train_df
# (Crucial step before feeding data into the model)
test_df = test_df[train_df.columns.drop('Survived')]

In [21]:
# Feature Selection & Data Split

from sklearn.model_selection import train_test_split

# Select engineered and cleaned features for the model
features = ['Pclass', 'Sex', 'Age', 'Embarked', 'Title', 'FamilyGroup']
X = train_df[features]
y = train_df['Survived']

# Split data into training and validation sets (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
# Feature Scaling

from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation data using the same scaler to avoid data leakage
X_val_scaled = scaler.transform(X_val)

---
# Model 1: k-Nearest Neighbors (k-NN)
---

In [23]:
# Hyperparameter Tuning & Model Training (GridSearchCV)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Define parameter grid for k-NN optimization
param_grid = {
    'n_neighbors': [5, 7, 9, 11, 13, 15, 17, 19, 21, 23],
    'weights': ['uniform', 'distance'],
    'p': [1, 2] # 1: Manhattan Distance, 2: Euclidean Distance
}

# Initialize the base k-NN model
knn_base = KNeighborsClassifier()

# Configure GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(estimator=knn_base, param_grid=param_grid, cv=5, scoring='accuracy')

# Execute grid search on scaled training data
print("Searching for the best parameter combination...")
grid_search.fit(X_train_scaled, y_train)

# Display optimization results
print("\n--- Grid Search Results ---")
print(f"Best Parameters: {grid_search.best_params_}")
print("-" * 30)

# Evaluate the optimized model on the validation set
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_val_scaled)

# Calculate final validation accuracy
accuracy = accuracy_score(y_val, y_pred)
print(f"Final Validation Accuracy: {accuracy * 100:.2f}%")

Searching for the best parameter combination...

--- Grid Search Results ---
Best Parameters: {'n_neighbors': 19, 'p': 1, 'weights': 'uniform'}
------------------------------
Final Validation Accuracy: 82.68%


In [24]:
# Final Prediction & Kaggle Submission

# Load the unlabelled testing dataset
test_df = pd.read_csv('../data/test.csv')

# --- Replicate Feature Engineering on Test Data ---
test_df['Title'] = test_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test_df['Title'] = test_df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
test_df['Title'] = test_df['Title'].replace('Mlle', 'Miss')
test_df['Title'] = test_df['Title'].replace('Ms', 'Miss')
test_df['Title'] = test_df['Title'].replace('Mme', 'Mrs')
test_df['Title'] = test_df['Title'].map(title_mapping)
test_df['Title'] = test_df['Title'].fillna(0)

test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1
test_df['FamilyGroup'] = test_df['FamilySize'].apply(group_family)
test_df['FamilyGroup'] = test_df['FamilyGroup'].map(family_mapping)

# --- Replicate Cleaning & Encoding on Test Data ---
test_df['Age'] = test_df['Age'].fillna(train_df['Age'].median()) 
test_df['Sex'] = test_df['Sex'].map({'male': 0, 'female': 1})
test_df['Embarked'] = test_df['Embarked'].fillna(train_df['Embarked'].mode()[0])
test_df['Embarked'] = test_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Select identical features and apply the fitted scaler
X_test_final = test_df[features]
X_test_scaled = scaler.transform(X_test_final)

# Predict survival utilizing the best k-NN model
predictions = best_knn.predict(X_test_scaled)

# Construct submission dataframe matching Kaggle formatting specifications
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': predictions
})

# Export predictions to CSV format
submission.to_csv('submission.csv', index=False)
print("Prediction completed! 'submission.csv' has been generated successfully.")
print("-" * 45)
print(submission.head(10))

Prediction completed! 'submission.csv' has been generated successfully.
---------------------------------------------
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0


---
# Model 2: Random Forest Classifier
---

In [25]:
# Random Forest Implementation

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Initialize the Random Forest model with 100 decision trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model using the unscaled training data
print("Training the Random Forest model...")
rf_model.fit(X_train, y_train)

# Predict survival outcomes on the validation set
rf_predictions = rf_model.predict(X_val)

# Calculate and display the final validation accuracy
rf_accuracy = accuracy_score(y_val, rf_predictions)
print("\n--- Random Forest Results ---")
print(f"Validation Accuracy: {rf_accuracy * 100:.2f}%")

Training the Random Forest model...

--- Random Forest Results ---
Validation Accuracy: 82.12%


In [26]:
# Tuning Random Forest with GridSearchCV

from sklearn.model_selection import GridSearchCV

# Define parameter grid for Random Forest optimization
rf_param_grid = {
    'n_estimators': [100, 200, 300],        # Number of trees in the forest
    'max_depth': [5, 8, 10, None],          # Maximum depth of the trees (prevents overfitting)
    'min_samples_split': [2, 5, 10],        # Minimum samples required to split an internal node
    'min_samples_leaf': [1, 2, 4]           # Minimum samples required to be at a leaf node
}

# Initialize the base Random Forest model
rf_base = RandomForestClassifier(random_state=42)

# Configure GridSearchCV with 5-fold cross-validation 
# (n_jobs=-1 utilizes all CPU cores for faster computation)
print("Tuning Random Forest... (This might take a minute)")
rf_grid_search = GridSearchCV(estimator=rf_base, param_grid=rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Execute grid search on the unscaled training data
rf_grid_search.fit(X_train, y_train)

# Display optimization results
print("\n--- Best Random Forest Parameters ---")
print(rf_grid_search.best_params_)

# Evaluate the optimized model on the validation set
best_rf = rf_grid_search.best_estimator_
best_rf_pred = best_rf.predict(X_val)
best_rf_accuracy = accuracy_score(y_val, best_rf_pred)

print("-" * 40)
print(f"Tuned RF Validation Accuracy: {best_rf_accuracy * 100:.2f}%")

Tuning Random Forest... (This might take a minute)

--- Best Random Forest Parameters ---
{'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}
----------------------------------------
Tuned RF Validation Accuracy: 81.56%


In [27]:
# Generate Kaggle Submission (Random Forest)

# Use the best Random Forest model to predict the test data
# Note: We use X_test_final (unscaled) because tree models do not require scaling
rf_predictions = best_rf.predict(X_test_final)

# Create a new submission dataframe
rf_submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': rf_predictions
})

# Save to a new CSV file to avoid overwriting the k-NN submission
rf_submission.to_csv('rf_submission.csv', index=False)

---
# Model 3: XGBoost Classifier
---

In [28]:
# Update Data Split with New Features

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# 1. Define the updated features (X) and target (y)
# Drop the target variable, identifiers, and explicitly remove the 'Name' string column.
X_updated = train_df.drop(['Survived', 'PassengerId', 'Name'], axis=1, errors='ignore')

# Define the target variable
y_updated = train_df['Survived'] 

X_test_final_updated = test_df.drop(['PassengerId', 'Name'], axis=1, errors='ignore')

# Failsafe Mechanism: Strictly exclude any remaining 'object' (string) data types.
X_updated = X_updated.select_dtypes(exclude=['object'])
X_test_final_updated = X_test_final_updated.select_dtypes(exclude=['object'])

# 2. Re-split the data (80% Train, 20% Validation)
X_train, X_val, y_train, y_val = train_test_split(X_updated, y_updated, test_size=0.2, random_state=42)

# XGBoost Implementation

# Initialize XGBoost model
# eval_metric='logloss' is added to prevent warning messages in newer versions
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric='logloss')

# Train the model
print("Training the XGBoost model...")
xgb_model.fit(X_train, y_train)

# Predict on validation set
xgb_predictions = xgb_model.predict(X_val)

# Calculate and display accuracy
xgb_accuracy = accuracy_score(y_val, xgb_predictions)
print("\n--- XGBoost Results ---")
print(f"Validation Accuracy: {xgb_accuracy * 100:.2f}%")

Training the XGBoost model...

--- XGBoost Results ---
Validation Accuracy: 83.80%


In [29]:
# Tuning XGBoost with GridSearchCV

from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Define parameter grid for XGBoost optimization
xgb_param_grid = {
    'n_estimators': [100, 200],          # Number of sequential trees to be modeled
    'max_depth': [3, 4, 5],              # Maximum depth of a tree (prevents overfitting)
    'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage used to update weights
    'subsample': [0.8, 1.0]              # Fraction of observations randomly sampled for each tree
}

# Initialize the base XGBoost model
xgb_base = XGBClassifier(random_state=42, eval_metric='logloss')

# Configure GridSearchCV with 5-fold cross-validation
print("Tuning XGBoost... (Unleashing the full power 🚀)")
xgb_grid_search = GridSearchCV(estimator=xgb_base, param_grid=xgb_param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Execute grid search on the training data
xgb_grid_search.fit(X_train, y_train)

# Display the best parameters discovered
print("\n--- Best XGBoost Parameters ---")
print(xgb_grid_search.best_params_)

# Evaluate the optimized model on the validation set
best_xgb = xgb_grid_search.best_estimator_
best_xgb_pred = best_xgb.predict(X_val)
best_xgb_accuracy = accuracy_score(y_val, best_xgb_pred)

print("-" * 40)
print(f"Tuned XGBoost Validation Accuracy: {best_xgb_accuracy * 100:.2f}%")

Tuning XGBoost... (Unleashing the full power 🚀)

--- Best XGBoost Parameters ---
{'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
----------------------------------------
Tuned XGBoost Validation Accuracy: 82.68%


In [30]:
# Pipeline Fix for Test Data

import pandas as pd

# 1. Reload raw test data just to borrow the original 'Cabin' and 'Ticket' safely
raw_test = pd.read_csv('../data/test.csv')

# 2. Force re-apply Feature Enrichment strictly to test_df
test_df['Deck'] = raw_test['Cabin'].fillna('U').apply(lambda x: str(x)[0])
test_df['Ticket_Freq'] = raw_test.groupby('Ticket')['Ticket'].transform('count')

# 3. Apply One-Hot Encoding to the newly created 'Deck' column
if 'Deck' in test_df.columns:
    test_df = pd.get_dummies(test_df, columns=['Deck'])

# 4. Strictly Align X_test_final_updated with X_train
# Ensure ANY missing columns (like Deck_T) are added and filled with 0
missing_cols = set(X_train.columns) - set(test_df.columns)
for col in missing_cols:
    test_df[col] = 0

# 5. Extract EXACTLY the same columns in the EXACT SAME ORDER as X_train
X_test_final_updated = test_df[X_train.columns]

print(f"Data rigorously aligned! Model expects {X_train.shape[1]} columns, Test data has {X_test_final_updated.shape[1]} columns.")

# Generate Final XGBoost Submission

# Predict using the perfectly aligned data
final_xgb_predictions = best_xgb.predict(X_test_final_updated)

# Create the submission DataFrame
xgb_submission = pd.DataFrame({
    "PassengerId": raw_test["PassengerId"],
    "Survived": final_xgb_predictions
})

# Export to CSV
xgb_submission.to_csv("xgb_submission.csv", index=False)

print("xgb_submission.csv successfully created! Ready for Kaggle")

Data rigorously aligned! Model expects 20 columns, Test data has 20 columns.
xgb_submission.csv successfully created! Ready for Kaggle
